# Prompt B — Debug Broken London House Price Pipeline

This notebook identifies and fixes all bugs in the provided pipeline,
runs the corrected version, and reports metrics before and after.

## 1. Original (Buggy) Pipeline

Run the original code as-is to capture the "before" metrics.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

# Prepare features
feature_cols = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                'rentEstimate', 'crime_total', 'census_denom_total']
X = df[feature_cols].fillna(0)
y = df['price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate (BUGGY — uses training data)
train_preds = model.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, train_preds))
mae = mean_absolute_error(y_train, train_preds)
r2 = r2_score(y_train, train_preds)

print('=== ORIGINAL (BUGGY) METRICS ===')
print(f"Test RMSE: £{rmse:,.0f}")
print(f"Test MAE:  £{mae:,.0f}")
print(f"Test R²:   {r2:.4f}")

=== ORIGINAL (BUGGY) METRICS ===
Test RMSE: £31,870
Test MAE:  £6,460
Test R²:   0.9988


## 2. Bug Identification

### Bug 1: Model evaluated on training data instead of test data

**What's wrong:** The evaluation section computes predictions on `X_train` and
compares them against `y_train`:

```python
train_preds = model.predict(X_train)  # <-- should be X_test
rmse = np.sqrt(mean_squared_error(y_train, train_preds))  # <-- should be y_test
```

The print statements label these as "Test" metrics, but they are actually
**training** metrics.

**Why it's a problem:** Random Forests can nearly memorise training data
(each tree sees bootstrap samples, so OOB error would be honest, but
in-bag predictions are nearly perfect). This produces massively inflated
R² and artificially low RMSE/MAE that do not reflect how the model would
perform on unseen data.

**Fix:** Use `model.predict(X_test)` and evaluate against `y_test`.

---

### Bug 2: `fillna(0)` is inappropriate for missing values

**What's wrong:** Missing values in features like `bedrooms`, `bathrooms`,
`floorAreaSqM`, `livingRooms`, and `rentEstimate` are filled with 0.

**Why it's a problem:**
- A property with 0 bedrooms, 0 bathrooms, or 0 m² floor area is nonsensical.
- For tree-based models, 0 creates an artificial category that splits
  differently from real values, pulling predictions toward prices of
  properties that genuinely have low feature values.
- `rentEstimate` filled with 0 would make a property look like it has
  zero rental value, drastically under-predicting its price (since
  rentEstimate is the dominant feature).

**Fix:** Use **median imputation** for numeric features, which preserves
the central tendency of each column.

## 3. Corrected Pipeline

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

# Prepare features — FIX Bug 2: median imputation instead of fillna(0)
feature_cols = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                'rentEstimate', 'crime_total', 'census_denom_total']
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['price']

print('Missing values after median imputation:')
print(X.isnull().sum())
print(f'\nMedian values used for imputation:')
print(df[feature_cols].median())

Missing values after median imputation:
bedrooms              0
bathrooms             0
floorAreaSqM          0
livingRooms           0
rentEstimate          0
crime_total           0
census_denom_total    0
dtype: int64

Median values used for imputation:
bedrooms                 2.000000
bathrooms                1.000000
floorAreaSqM            84.000000
livingRooms              1.000000
rentEstimate          2600.000000
crime_total           6075.000000
census_denom_total     666.058442
dtype: float64


In [3]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')

Training set: 334,048 rows
Test set:     83,513 rows


In [4]:
# Train model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate — FIX Bug 1: predict on TEST data, compare against y_test
test_preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
mae = mean_absolute_error(y_test, test_preds)
r2 = r2_score(y_test, test_preds)

print('=== CORRECTED METRICS (Test Set) ===')
print(f"Test RMSE: £{rmse:,.0f}")
print(f"Test MAE:  £{mae:,.0f}")
print(f"Test R²:   {r2:.4f}")

=== CORRECTED METRICS (Test Set) ===
Test RMSE: £66,194
Test MAE:  £15,087
Test R²:   0.9949


## 4. Impact of Each Bug on Reported Results

### Bug 1: Training-set evaluation

- **Effect:** R² was inflated close to 1.0 (Random Forest memorises
  training data), and RMSE/MAE were artificially low.
- **Severity:** Critical — the reported metrics gave a completely
  misleading picture of model generalisation. A model could appear
  near-perfect while actually performing much worse on unseen data.

### Bug 2: fillna(0)

- **Effect:** Properties with missing bedrooms/bathrooms/floor area
  were treated as having 0 of those features, causing the model to
  learn incorrect associations. Most importantly, ~1,100 properties
  with missing `rentEstimate` (the dominant predictor) were assigned
  £0 rent, making them look like worthless properties and distorting
  predictions.
- **Severity:** Moderate — the effect is dampened because the majority
  of rows have complete data, but predictions for affected rows would
  be substantially wrong.

## 5. Summary Comparison

In [5]:
print('Bug Summary')
print('=' * 70)
print(f'{"Bug":>4} | {"Description":<45} | {"Severity"}')
print('-' * 70)
print(f'{"1":>4} | {"Evaluated on training set, not test set":<45} | Critical')
print(f'{"2":>4} | {"fillna(0) instead of median imputation":<45} | Moderate')
print()
print('Note: The print labels said "Test" but the code computed training')
print('metrics — this is part of Bug 1 (misleading output).')

Bug Summary
 Bug | Description                                   | Severity
----------------------------------------------------------------------
   1 | Evaluated on training set, not test set       | Critical
   2 | fillna(0) instead of median imputation        | Moderate

Note: The print labels said "Test" but the code computed training
metrics — this is part of Bug 1 (misleading output).
